# Self-Training (ST) for Few-Shot Disaster Tweet Classification

This notebook runs **Self-Training (ST)** experiments using a BERT-based model (BERTweet) for classifying disaster-related tweets into 10 humanitarian categories.

## Algorithm Overview

Self-Training is a semi-supervised learning approach that leverages a small set of labeled data together with a large pool of unlabeled data:

1. **Supervised fine-tuning (base model):** A pre-trained BERTweet model is fine-tuned on the small labeled dataset. To reduce sensitivity to random initialization, `N_base` independent runs are performed and the best model (by validation macro-F1) is selected.

2. **Iterative pseudo-labeling:** For each self-training iteration:
   - A subset of unlabeled examples (`sample_size`) is drawn from the unlabeled pool.
   - The model assigns pseudo-labels to these examples. With the `"uniform"` sampling scheme, pseudo-labeled instances are selected uniformly at random (no uncertainty estimation).
   - A new training set is formed by combining the original labeled data with the selected pseudo-labeled data (`unsup_size` instances).
   - The model is re-trained on this combined set, with a weighted loss: 50% supervised + 50% unsupervised. Confidence-weighted loss reweighting is controlled by `alpha`.
   - Early stopping on validation macro-F1 prevents overfitting.

3. **Evaluation:** The best checkpoint is evaluated on a held-out test set, reporting macro-F1 and Expected Calibration Error (ECE with `n_bins=10`).

## Humanitarian Categories

The full label space contains up to 10 classes, but **not every disaster dataset has all 10**. Some events may only have 7, 8, or 9 classes depending on the types of tweets observed. The notebook automatically detects the actual number of classes per dataset.

| ID | Category |
|---|---|
| 0 | Caution and advice |
| 1 | Displaced people and evacuations |
| 2 | Infrastructure and utility damage |
| 3 | Injured or dead people |
| 4 | Missing or found people |
| 5 | Not humanitarian |
| 6 | Other relevant information |
| 7 | Requests or urgent needs |
| 8 | Rescue, volunteering, or donation effort |
| 9 | Sympathy and support |

## Datasets

Each disaster folder under `data/` contains:
- `labeled_{k}_set{s}.tsv` — few-shot labeled splits (k = 5, 10, 25, 50 per class; s = 1, 2, 3)
- `unlabeled_{k}_set{s}.tsv` — corresponding unlabeled pools
- `{disaster}_dev.tsv` — validation split
- `{disaster}_test.tsv` — test split

## Key Hyperparameters

| Parameter | Default | Description |
|---|---|---|
| `sample_scheme` | `"uniform"` | Sampling strategy for pseudo-label selection (uniform = standard ST) |
| `sup_epochs` | 18 | Max epochs for supervised fine-tuning (with early stopping, patience=3) |
| `unsup_epochs` | 12 | Number of self-training iterations |
| `N_base` | 3 | Number of random initializations for base model selection |
| `T` | 7 | Number of MC Dropout forward passes (not used for uniform scheme) |
| `alpha` | 0.1 | Confidence loss reweighting factor |
| `sample_size` | 1800 | Unlabeled instances sampled for uncertainty evaluation per iteration |
| `unsup_size` | 1000 | Pseudo-labeled instances used per self-training iteration |
| `sup_batch_size` | 16 | Batch size for supervised training |
| `unsup_batch_size` | 64 | Batch size for self-training on pseudo-labeled data |

## Setup

Import dependencies and configure the environment. The `PYTHONHASHSEED` environment variable must be set for reproducibility — it is used as the global seed throughout the pipeline.

In [ ]:
import os
import sys
import json
import logging
import numpy as np
import pandas as pd
import random

# Set seeds BEFORE importing torch/transformers
os.environ["PYTHONHASHSEED"] = "42"
GLOBAL_SEED = int(os.environ["PYTHONHASHSEED"])
random.seed(GLOBAL_SEED)
np.random.seed(GLOBAL_SEED)

import torch
torch.manual_seed(GLOBAL_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(GLOBAL_SEED)

# Add the project root to the path so we can import project modules
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from transformers import AutoConfig, AutoTokenizer
from custom_dataset import CustomDataset_tracked, CustomDataset
from ust import train_model

# Logging
logger = logging.getLogger("UST")
logging.basicConfig(level=logging.INFO)

print(f"Global seed: {GLOBAL_SEED}")
print(f"Project root: {PROJECT_ROOT}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## Helper Functions

Define the label mapping and dataset loading utilities. These mirror what `run_ust.py` does but are defined here so the notebook is self-contained.

**Important:** Not every disaster has all 10 classes. `detect_classes()` scans the train, dev, and test splits to build the actual label-to-id mapping for each disaster, so the model only predicts the classes that are actually present.

In [ ]:
# Full 10-class humanitarian label mapping (superset)
FULL_LABEL_TO_ID = {
    "caution_and_advice": 0,
    "displaced_people_and_evacuations": 1,
    "infrastructure_and_utility_damage": 2,
    "injured_or_dead_people": 3,
    "missing_or_found_people": 4,
    "not_humanitarian": 5,
    "other_relevant_information": 6,
    "requests_or_urgent_needs": 7,
    "rescue_volunteering_or_donation_effort": 8,
    "sympathy_and_support": 9,
}


def detect_classes(disaster, train_file, data_root="data"):
    """Detect the actual classes present in a disaster dataset.
    
    Scans train, dev, and test TSV files to collect all unique class labels,
    then builds a contiguous label-to-id mapping (0, 1, 2, ...).
    
    Returns:
        label_to_id (dict): mapping from class name to integer id
        n_classes (int): number of unique classes
    """
    base = os.path.join(data_root, disaster)
    files = [
        os.path.join(base, f"labeled_{train_file}.tsv"),
        os.path.join(base, f"{disaster}_dev.tsv"),
        os.path.join(base, f"{disaster}_test.tsv"),
    ]
    all_labels = set()
    for f in files:
        if os.path.exists(f):
            df = pd.read_csv(f, sep="\t")
            all_labels.update(df["class_label"].dropna().unique())

    # Build a contiguous mapping, preserving the canonical order from FULL_LABEL_TO_ID
    label_to_id = {}
    idx = 0
    for label in FULL_LABEL_TO_ID:
        if label in all_labels:
            label_to_id[label] = idx
            idx += 1

    return label_to_id, len(label_to_id)


def get_dataset(path, tokenizer, label_to_id, labeled=True):
    """Load a TSV file into a CustomDataset_tracked instance."""
    df = pd.read_csv(path, sep="\t")
    text_list = []
    labels_list = []
    ids_list = []
    for _, row in df.iterrows():
        if pd.isna(row["tweet_text"]):
            continue
        text_list.append(row["tweet_text"])
        labels_list.append(label_to_id[row["class_label"]])
        ids_list.append(row["tweet_id"])
    return CustomDataset_tracked(text_list, labels_list, ids_list, tokenizer, labeled=labeled)


def load_disaster_datasets(disaster, train_file, tokenizer, label_to_id, data_root="data"):
    """Load train, dev, test, and unlabeled datasets for a given disaster."""
    base = os.path.join(data_root, disaster)
    ds_train = get_dataset(os.path.join(base, f"labeled_{train_file}.tsv"), tokenizer, label_to_id)
    ds_dev = get_dataset(os.path.join(base, f"{disaster}_dev.tsv"), tokenizer, label_to_id)
    ds_test = get_dataset(os.path.join(base, f"{disaster}_test.tsv"), tokenizer, label_to_id)
    ds_unlabeled = get_dataset(os.path.join(base, f"unlabeled_{train_file}.tsv"), tokenizer, label_to_id, labeled=False)
    print(f"  Train: {len(ds_train)} | Dev: {len(ds_dev)} | Test: {len(ds_test)} | Unlabeled: {len(ds_unlabeled)}")
    return ds_train, ds_dev, ds_test, ds_unlabeled


print(f"Full label space: {len(FULL_LABEL_TO_ID)} classes")
print(f"Labels: {list(FULL_LABEL_TO_ID.keys())}")

## Configuration

Set the model checkpoint, dropout rates, and other hyperparameters. For **standard Self-Training**, we use `sample_scheme = "uniform"`, which randomly selects pseudo-labeled instances without uncertainty-based filtering.

All other hyperparameters use their default values as defined in the project.

In [ ]:
# Model checkpoint
PT_TEACHER_CHECKPOINT = "vinai/bertweet-base"

# Dropout configuration
HIDDEN_DROPOUT_PROB = 0.3
ATTENTION_PROBS_DROPOUT_PROB = 0.3
DENSE_DROPOUT = 0.5

# Self-Training hyperparameters
SAMPLE_SCHEME = "uniform"        # Standard ST — uniform pseudo-label selection
SUP_EPOCHS = 18                  # Max supervised fine-tuning epochs (early stopping patience=3)
UNSUP_EPOCHS = 12                # Number of self-training iterations
N_BASE = 3                       # Random initializations for base model selection
T = 7                            # MC Dropout passes (unused for uniform scheme)
ALPHA = 0.1                      # Confidence loss reweighting factor
SAMPLE_SIZE = 1800               # Unlabeled instances sampled per ST iteration
UNSUP_SIZE = 1000                # Pseudo-labeled instances used per ST iteration
SUP_BATCH_SIZE = 16
UNSUP_BATCH_SIZE = 64

# Training data split
TRAIN_FILE = "5_set1"            # 5 labeled examples per class, set 1

# Results output directory (separate from data/)
RESULTS_DIR = os.path.join(PROJECT_ROOT, "results")
RESULTS_FILE = "st_uniform_result"

# Build model config with custom dropout
cfg = AutoConfig.from_pretrained(PT_TEACHER_CHECKPOINT)
cfg.hidden_dropout_prob = HIDDEN_DROPOUT_PROB
cfg.attention_probs_dropout_prob = ATTENTION_PROBS_DROPOUT_PROB

# Initialize tokenizer (shared across all experiments)
tokenizer = AutoTokenizer.from_pretrained(PT_TEACHER_CHECKPOINT)

print("Configuration loaded.")
print(f"  Model: {PT_TEACHER_CHECKPOINT}")
print(f"  Sample scheme: {SAMPLE_SCHEME}")
print(f"  Train file pattern: labeled_{TRAIN_FILE}.tsv / unlabeled_{TRAIN_FILE}.tsv")
print(f"  Results will be saved to: {RESULTS_DIR}/{{disaster}}/")
print(f"  Supervised epochs: {SUP_EPOCHS}, ST iterations: {UNSUP_EPOCHS}, N_base: {N_BASE}")
print(f"  Note: n_classes will be detected per disaster (not all have 10 classes)")

---

## Part 1: Quick Sanity Check — Single Disaster

Before running on all datasets, we verify the pipeline works end-to-end on a **single disaster** (`california_wildfires_2018`) with the smallest labeled split (`5_set1` = 5 examples per class = 50 total).

This cell should complete relatively quickly and confirms that:
- Data loading works correctly
- The base model trains and selects the best initialization
- Self-training iterations run without errors
- Test evaluation produces valid F1 and ECE scores

If this cell runs successfully, you can proceed to the full experiment below.

In [ ]:
# --- Sanity check: single disaster ---
SANITY_DISASTER = "california_wildfires_2018"
DATA_ROOT = os.path.join(PROJECT_ROOT, "data")

# Detect the actual classes for this disaster
label_to_id, n_classes = detect_classes(SANITY_DISASTER, TRAIN_FILE, data_root=DATA_ROOT)
print(f"Loading data for: {SANITY_DISASTER}")
print(f"  Detected {n_classes} classes: {list(label_to_id.keys())}")

ds_train, ds_dev, ds_test, ds_unlabeled = load_disaster_datasets(
    SANITY_DISASTER, TRAIN_FILE, tokenizer, label_to_id, data_root=DATA_ROOT
)

print(f"\nStarting Self-Training on {SANITY_DISASTER}...")
print("=" * 60)

train_model(
    ds_train, ds_dev, ds_test, ds_unlabeled,
    PT_TEACHER_CHECKPOINT, cfg,
    model_dir=SANITY_DISASTER,
    sup_batch_size=SUP_BATCH_SIZE,
    unsup_batch_size=UNSUP_BATCH_SIZE,
    unsup_size=UNSUP_SIZE,
    sample_size=SAMPLE_SIZE,
    sample_scheme=SAMPLE_SCHEME,
    T=T,
    alpha=ALPHA,
    sup_epochs=SUP_EPOCHS,
    unsup_epochs=UNSUP_EPOCHS,
    N_base=N_BASE,
    dense_dropout=DENSE_DROPOUT,
    attention_probs_dropout_prob=ATTENTION_PROBS_DROPOUT_PROB,
    hidden_dropout_prob=HIDDEN_DROPOUT_PROB,
    results_file=RESULTS_FILE,
    results_dir=RESULTS_DIR,
    temp_scaling=False,
    ls=0.0,
    n_classes=n_classes,
)

# Display results
results_path = os.path.join(RESULTS_DIR, SANITY_DISASTER, f"{RESULTS_FILE}.txt")
if os.path.exists(results_path):
    with open(results_path) as f:
        results = json.load(f)
    print(f"\nResults for {SANITY_DISASTER}:")
    print(json.dumps(results, indent=2))
else:
    print(f"\nWarning: results file not found at {results_path}")

print("\nSanity check complete.")

---

## Part 2: Full Experiment — All Disasters

Now we run Self-Training across **all 10 disaster datasets**. For each disaster, the pipeline:

1. Loads the labeled, dev, test, and unlabeled splits
2. Trains `N_base=3` base models and selects the best by validation F1
3. Runs `unsup_epochs=12` self-training iterations with uniform pseudo-label selection
4. Evaluates on the test set and saves results to `results/{disaster}/st_uniform_result.txt`

**Note:** This will take a significant amount of time depending on your hardware. Each disaster involves multiple epochs of BERT fine-tuning plus self-training iterations. Progress is logged for each disaster.

In [ ]:
# List all disaster datasets
ALL_DISASTERS = sorted([
    d for d in os.listdir(DATA_ROOT)
    if os.path.isdir(os.path.join(DATA_ROOT, d))
])

print(f"Found {len(ALL_DISASTERS)} disaster datasets:")
for i, d in enumerate(ALL_DISASTERS, 1):
    print(f"  {i}. {d}")

In [ ]:
# --- Run ST on all disasters ---
all_results = {}

for i, disaster in enumerate(ALL_DISASTERS, 1):
    print(f"\n{'=' * 60}")
    print(f"[{i}/{len(ALL_DISASTERS)}] {disaster}")
    print(f"{'=' * 60}")

    # Check that the required files exist
    labeled_path = os.path.join(DATA_ROOT, disaster, f"labeled_{TRAIN_FILE}.tsv")
    unlabeled_path = os.path.join(DATA_ROOT, disaster, f"unlabeled_{TRAIN_FILE}.tsv")
    if not os.path.exists(labeled_path) or not os.path.exists(unlabeled_path):
        print(f"  Skipping {disaster}: missing labeled or unlabeled file for split '{TRAIN_FILE}'")
        continue

    # Detect actual classes for this disaster
    label_to_id, n_classes = detect_classes(disaster, TRAIN_FILE, data_root=DATA_ROOT)
    print(f"  Detected {n_classes} classes: {list(label_to_id.keys())}")

    # Load datasets
    ds_train, ds_dev, ds_test, ds_unlabeled = load_disaster_datasets(
        disaster, TRAIN_FILE, tokenizer, label_to_id, data_root=DATA_ROOT
    )

    # Run self-training
    train_model(
        ds_train, ds_dev, ds_test, ds_unlabeled,
        PT_TEACHER_CHECKPOINT, cfg,
        model_dir=disaster,
        sup_batch_size=SUP_BATCH_SIZE,
        unsup_batch_size=UNSUP_BATCH_SIZE,
        unsup_size=UNSUP_SIZE,
        sample_size=SAMPLE_SIZE,
        sample_scheme=SAMPLE_SCHEME,
        T=T,
        alpha=ALPHA,
        sup_epochs=SUP_EPOCHS,
        unsup_epochs=UNSUP_EPOCHS,
        N_base=N_BASE,
        dense_dropout=DENSE_DROPOUT,
        attention_probs_dropout_prob=ATTENTION_PROBS_DROPOUT_PROB,
        hidden_dropout_prob=HIDDEN_DROPOUT_PROB,
        results_file=RESULTS_FILE,
        results_dir=RESULTS_DIR,
        temp_scaling=False,
        ls=0.0,
        n_classes=n_classes,
    )

    # Collect results
    results_path = os.path.join(RESULTS_DIR, disaster, f"{RESULTS_FILE}.txt")
    if os.path.exists(results_path):
        with open(results_path) as f:
            result = json.load(f)
        all_results[disaster] = result
        all_results[disaster]["n_classes"] = n_classes
        f1 = result.get("Best ST model", {}).get("F1 before temp scaling", "N/A")
        ece = result.get("Best ST model", {}).get("ECE before temp scaling", "N/A")
        print(f"  Test F1: {f1} | ECE: {ece}")
    else:
        print(f"  Warning: results file not found.")

print(f"\n{'=' * 60}")
print("All experiments complete.")

## Results Summary

Aggregate results from all disasters into a table for easy comparison.

In [ ]:
# Build summary table
summary_rows = []
for disaster, result in all_results.items():
    best = result.get("Best ST model", {})
    summary_rows.append({
        "Disaster": disaster,
        "Classes": result.get("n_classes", "N/A"),
        "F1 (macro)": best.get("F1 before temp scaling", "N/A"),
        "ECE": best.get("ECE before temp scaling", "N/A"),
    })

if summary_rows:
    df_summary = pd.DataFrame(summary_rows)
    # Convert to numeric for mean computation
    df_summary["F1 (macro)"] = pd.to_numeric(df_summary["F1 (macro)"], errors="coerce")
    df_summary["ECE"] = pd.to_numeric(df_summary["ECE"], errors="coerce")

    print(df_summary.to_string(index=False))
    print(f"\n{'─' * 50}")
    print(f"Mean F1: {df_summary['F1 (macro)'].mean():.4f}")
    print(f"Mean ECE: {df_summary['ECE'].mean():.4f}")
else:
    print("No results collected. Check the experiment logs above for errors.")